In [3]:
! pip install pymilvus
! pip install sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.8/344.8 kB 6.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 30.5 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.6 which is incompatible.


In [10]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
ZILLIZ_TOKEN = user_secrets.get_secret("zilliz_api_key")
ZILLIZ_ENDPOINT = user_secrets.get_secret("Zilliz_endpoint")


In [4]:
import pandas as pd
from pymilvus import MilvusClient
from sentence_transformers import SentenceTransformer

In [11]:
# 1. Load the data
# Note: In Kaggle, the path includes the dataset name
FILE_PATH = '/kaggle/input/datasets/ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data/HDFS_2k/HDFS_2k.log_structured.csv'
df = pd.read_csv(FILE_PATH)
print(df.shape)
print(df.head(5))

(2000, 9)
   LineId   Date    Time  Pid Level                     Component  \
0       1  81109  203615  148  INFO  dfs.DataNode$PacketResponder   
1       2  81109  203807  222  INFO  dfs.DataNode$PacketResponder   
2       3  81109  204005   35  INFO              dfs.FSNamesystem   
3       4  81109  204015  308  INFO  dfs.DataNode$PacketResponder   
4       5  81109  204106  329  INFO  dfs.DataNode$PacketResponder   

                                             Content EventId  \
0  PacketResponder 1 for block blk_38865049064139...     E10   
1  PacketResponder 0 for block blk_-6952295868487...     E10   
2  BLOCK* NameSystem.addStoredBlock: blockMap upd...      E6   
3  PacketResponder 2 for block blk_82291938032499...     E10   
4  PacketResponder 2 for block blk_-6670958622368...     E10   

                                       EventTemplate  
0  PacketResponder <*> for block blk_<*> terminating  
1  PacketResponder <*> for block blk_<*> terminating  
2  BLOCK* NameSystem.addS

In [12]:
def smart_truncate(text, max_len=2980):
    if len(text) <= max_len:
        return text
    # Keep the first 1000 chars and the last 1000 chars
    return f"{text[:1000]}...[TRUNCATED]...{text[-1000:]}"


In [9]:
# client = MilvusClient(
#     uri=ZILLIZ_ENDPOINT,
#     token=ZILLIZ_TOKEN
# )

# client.delete(
#     collection_name="logs_collection",
#     filter="id >= 0"
# )

# print("Data cleared from logs_collection (Schema preserved).")

Data cleared from logs_collection (Schema preserved).


In [13]:
# 2. Setup Zilliz Client
client = MilvusClient(
    uri=ZILLIZ_ENDPOINT,
    token=ZILLIZ_TOKEN
)

partition_name = "hdfs"

# Create the partition if it doesn't exist
if not client.has_partition(collection_name="logs_collection", partition_name=partition_name):
    client.create_partition(
        collection_name="logs_collection", 
        partition_name=partition_name
    )
    print(f"Partition '{partition_name}' created.")

# 3. Load Embedding Model (BERT-based)
model = SentenceTransformer('all-MiniLM-L6-v2')

# 4. Prepare Data for Zilliz
data = []
print(f"Processing {len(df)} logs...")

# We process all 2,000 logs 
for idx, row in df.iterrows():
    
    vector = model.encode(row['Content']).tolist()
    
    # Combine Date and Time for the timestamp field
    full_timestamp = f"{row['Date']} {row['Time']}"

    truncated_content = smart_truncate(str(row['Content']))
    
    data.append({
        "vector": vector,
        "timestamp": str(full_timestamp),
        "log_level": str(row['Level']),
        "message": truncated_content
    })

# 5. Insert into Zilliz
client.insert(collection_name="logs_collection", data=data, partition_name=partition_name)

print("✅ Ingestion Complete! Data is now in Zilliz.")

Partition 'hdfs' created.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Processing 2000 logs...
✅ Ingestion Complete! Data is now in Zilliz.


In [15]:
stats = client.get_collection_stats(collection_name="logs_collection")
print(f"Total Collection Count (Approx): {stats['row_count']}")

# 2. Try to 'Query' the partition specifically
# This forces the database to look inside the 'toronto_logs' folder
res = client.query(
    collection_name="logs_collection",
    filter="id >= 0",
    partition_names=["hdfs"], # Look ONLY here
    limit=5,
    output_fields=["message"]
)

if res:
    print(f"✅ Success! Found {len(res)} logs in 'hdfs' partition.")
    print(f"Sample: {res[0]['message']}")
else:
    print("❌ Partition is actually empty. Ingestion might have failed.")

Total Collection Count (Approx): 2000
✅ Success! Found 5 logs in 'hdfs' partition.
Sample: PacketResponder 1 for block blk_38865049064139660 terminating
